# NB01 — CN-WaTEM/SEDEM: event runoff experiment
Standalone extension of NB00, with shared validated Langegracht inputs and separate outputs. Synthetic storm and illustrative CN assumptions; not a calibrated hydrological model. Uses WaTEM/SEDEM 5.0.3 and the same pinned wrapper. CN-WS couples event sediment totals to runoff hydrographs; it does not dynamically recompute detachment from runoff intensity. See https://github.com/watem-sedem/watem-sedem/blob/5.0.3/docs/cn-ws.rst.

## NB01.01 — Drive and runtime

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, shutil, subprocess, json, hashlib, time, platform
from datetime import datetime, timezone
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Watem')
DATA = ROOT / 'data/tutorial/langegracht'
OUTPUT = ROOT / 'outputs/cn_tutorial'
SOFTWARE = ROOT / 'software'
for p in (DATA.parent, OUTPUT, SOFTWARE): p.mkdir(parents=True, exist_ok=True)
REV = '226a7c06ae836f9736c1a88b74b5346033a2a8d6'
REPO = Path('/content/pywatemsedem')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
WORK = Path('/content/watem_cn_runs') / RUN_ID
WORK.mkdir(parents=True)
def run(cmd, **kwargs):
    print('+', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(list(map(str,cmd)), check=True, **kwargs)
print(ROOT, sys.version, platform.machine())

## NB01.02 — Install and verify same software as NB00

In [ ]:
import urllib.request
os_release = dict(line.strip().split('=',1) for line in Path('/etc/os-release').read_text().splitlines() if '=' in line)
codename = os_release.get('VERSION_CODENAME','').strip('"')
assert platform.machine() == 'x86_64', 'Expected x86_64 Colab runtime'
assert codename in {'jammy','noble','focal','bookworm'}, f'Unsupported OS for pinned SAGA release: {codename}'
run(['apt-get','update','-qq'])
run(['apt-get','install','-y','git-lfs','saga'])
run(['git','lfs','install'])
def download(url, path):
    if not path.exists():
        tmp = path.with_suffix(path.suffix+'.part')
        urllib.request.urlretrieve(url, tmp)
        tmp.replace(path)
    return path
saga_name = f'saga-vpo_2025.6.11-1-ge95f136.{codename}_amd64.deb'
saga_deb = download('https://github.com/DOV-Vlaanderen/saga-watem/releases/download/2025.6.11/'+saga_name, SOFTWARE/saga_name)
local_deb = Path('/tmp') / saga_name
shutil.copy2(saga_deb, local_deb)
run(['apt-get','install','-y',str(local_deb)])
ws = download('https://github.com/watem-sedem/watem-sedem/releases/download/5.0.3/watem_sedem', SOFTWARE/'watem_sedem_5.0.3')
binary = Path('/usr/local/bin/watem_sedem')
shutil.copy2(ws,binary); binary.chmod(0o755)
os.environ['WATEMSEDEM'] = str(binary.parent)
# saga-vpo contains plugins, not the SAGA command-line program.
# Ubuntu's saga package supplies saga_cmd.
saga_cmd = shutil.which('saga_cmd')
if not saga_cmd:
    run(['dpkg','-L','saga'])
    raise RuntimeError('saga_cmd missing after installing saga')
assert shutil.which('watem_sedem'), 'watem_sedem missing from PATH'
print('SAGA executable:', saga_cmd)
print('WaTEM executable:', shutil.which('watem_sedem'))
plugin_files = subprocess.check_output(['dpkg','-L','saga-vpo'],text=True).splitlines()
print('Installed SAGA plugin libraries:', [p for p in plugin_files if p.endswith('.so')])
run(['saga_cmd','--version'])
for args in (['saga_cmd','topology'], ['saga_cmd','watem']):
    p = subprocess.run(args,capture_output=True,text=True)
    msg = p.stdout+p.stderr
    print(msg[:3500])
    assert 'error: select a library' not in msg.lower() and 'cannot load' not in msg.lower(), f'Missing/incompatible SAGA library: {args}'
run(['ldd',str(binary)])
if not REPO.exists():
    run(['git','clone','https://github.com/watem-sedem/pywatemsedem.git',REPO],
        env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run(['git','-C',REPO,'checkout',REV],env={**os.environ,'GIT_LFS_SKIP_SMUDGE':'1'})
run([sys.executable,'-m','pip','install',str(REPO)])
import pywatemsedem
print('pywatemsedem',pywatemsedem.__version__)

## NB01.03 — Reuse checksum-verified tutorial data

In [ ]:
required = ['dtm.tif','K3.tif','basemap_landuse.tif','userchoices.ini']
stems = ['catchm_langegracht','river','infrastructure','roads','water','parcels','grass_strips','buffers','force_routing','outlets']
source = REPO/'tests/data'
names = sorted(p.name for p in source.iterdir() if p.is_file() and (p.name in required or p.stem in stems))
assert set(required).issubset(names)
for stem in stems:
    assert all(stem+ext in names for ext in ('.shp','.shx','.dbf','.prj')), stem
def expected_hash(name):
    raw = subprocess.check_output(['git','-C',str(REPO),'show',f'{REV}:tests/data/{name}'])
    if raw.startswith(b'version https://git-lfs'):
        return next(x.split(':',1)[1] for x in raw.decode().splitlines() if x.startswith('oid sha256:'))
    return hashlib.sha256(raw).hexdigest()
hashes = {name:expected_hash(name) for name in names}
import urllib.request, urllib.error
from concurrent.futures import ThreadPoolExecutor, as_completed

def valid_file(path, name):
    return path.is_file() and hashlib.sha256(path.read_bytes()).hexdigest() == hashes[name]

def valid(folder):
    return all(valid_file(folder/n,n) for n in names)

DATA.mkdir(parents=True,exist_ok=True)
missing = [n for n in names if not valid_file(DATA/n,n)]
print(f'Valid cached files: {len(names)-len(missing)}/{len(names)}')
if missing:
    # Git LFS pull can fail at a detached HEAD or due to upstream LFS service.
    # Capture stderr in the notebook, rather than only showing CalledProcessError.
    pending = [n for n in missing if not valid_file(source/n,n)]
    if pending:
        p = subprocess.run(['git','-C',str(REPO),'lfs','pull','origin',
                            '--include='+','.join('tests/data/'+n for n in pending)],
                           capture_output=True,text=True)
        print('Git LFS return code:',p.returncode)
        print(p.stdout[-4000:])
        print(p.stderr[-6000:])
    def retrieve(name):
        if valid_file(DATA/name,name):
            return name,'cached'
        if valid_file(source/name,name):
            shutil.copy2(source/name,DATA/name)
            return name,'git-lfs'
        # GitHub media endpoint serves actual LFS objects rather than pointers.
        # Always require the exact SHA256 from the pinned upstream Git tree.
        errors=[]
        tmp=DATA/(name+'.part')
        for base in (
            f'https://media.githubusercontent.com/media/watem-sedem/pywatemsedem/{REV}/tests/data/',
            f'https://github.com/watem-sedem/pywatemsedem/raw/{REV}/tests/data/',
            f'https://raw.githubusercontent.com/watem-sedem/pywatemsedem/{REV}/tests/data/',
        ):
            for attempt in range(2):
                try:
                    req=urllib.request.Request(base+name,headers={'User-Agent':'Watem-Colab/1.0'})
                    with urllib.request.urlopen(req,timeout=60) as response, tmp.open('wb') as dst:
                        shutil.copyfileobj(response,dst)
                    if not valid_file(tmp,name):
                        raise ValueError('SHA256 mismatch (possibly a Git LFS pointer)')
                    tmp.replace(DATA/name)
                    return name,'direct download'
                except Exception as e:
                    errors.append(f'{base}: {type(e).__name__}: {e}')
                    tmp.unlink(missing_ok=True)
                    if attempt==0: time.sleep(2)
        raise RuntimeError(name+': '+' | '.join(errors))
    failures=[]
    with ThreadPoolExecutor(max_workers=4) as pool:
        futures={pool.submit(retrieve,name):name for name in missing}
        for f in as_completed(futures):
            try:
                print(*f.result())
            except Exception as e:
                failures.append(str(e))
                print('FAILED:',e)
    if failures:
        raise RuntimeError(f'{len(failures)} data downloads failed. Valid files are retained on Drive. See HTTP/LFS diagnostics above.')
assert valid(DATA), 'Tutorial data checksum validation failed'
(DATA/'source_manifest.json').write_text(json.dumps({'revision':REV,'sha256':hashes},indent=2))
inputdata_folder = WORK/'inputs'
inputdata_folder.mkdir(exist_ok=True)
for n in names: shutil.copy2(DATA/n,inputdata_folder/n)
print(f'Validated {len(names)} files in {DATA}')
print('Local working data:', inputdata_folder)

## NB01.04 — Initialize catchment and reference settings

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import matplotlib.pyplot as plt
from IPython.display import display
from copy import deepcopy
import configparser
from pywatemsedem.catchment import Catchment
catchment = Catchment('langegracht',inputdata_folder/'catchm_langegracht.shp',
    inputdata_folder/'dtm.tif',20,31370,-9999,results_folder=WORK)
catchment.kfactor=inputdata_folder/'K3.tif'
catchment.landuse=inputdata_folder/'basemap_landuse.tif'
catchment.vct_river=inputdata_folder/'river.shp'
catchment.vct_infrastructure_buildings=inputdata_folder/'infrastructure.shp'
catchment.vct_infrastructure_roads=inputdata_folder/'roads.shp'
catchment.vct_water=inputdata_folder/'water.shp'
catchment.dtm.plot()
catchment.segments.plot(nodata=0)
from pywatemsedem.choices import Choices, Options, Parameters, Extensions, ExtensionsParameters, Output
from pywatemsedem.scenario import Scenario
default_choices=inputdata_folder/'userchoices.ini'
options=Options(); options.read_values_from_ini(default_choices)
parameters=Parameters(); parameters.read_values_from_ini(default_choices)
extensions=Extensions(); extensions.read_values_from_ini(default_choices)
output=Output(); output.read_values_from_ini(default_choices)
extensionparameters=ExtensionsParameters(extensions)
extensionparameters.apply_defaults()
for p in (extensionparameters.ktc_low,extensionparameters.ktc_high,extensionparameters.ktc_limit):
    p.read_value_from_ini(default_choices)
choices=Choices(options,parameters,extensions,extensionparameters,output)
assert not choices.extensions.curve_number.value, 'NB00 expects standard erosion model'
print(default_choices.read_text())

## NB01.05 — Synthetic storm and event settings
Rainfall file contains incremental mm at 10-minute intervals, first row zero. Event R is computed using a transparent Brown–Foster energy approximation and 30-minute maximum intensity (not annual R). Neutral alpha/beta isolate CN effects initially. 1 min routing step and 0.2 m/s meet the documented Courant criterion at 20 m. Endtime 12 h; inspect hydrograph tails.

In [ ]:
rain=np.array([0,2,4,7,9,8,5,3,2,0],dtype=float)
minutes=np.arange(len(rain))*10
rainfile=WORK/'storm.txt'
np.savetxt(rainfile,np.column_stack([minutes,rain]),fmt=['%d','%.3f'],delimiter='\t')
intensity=rain[1:]*6
energy=np.sum(0.29*(1-0.72*np.exp(-0.05*intensity))*rain[1:])
i30=np.max(np.convolve(rain[1:],np.ones(3),mode='valid'))*2
R_EVENT=float(energy*i30)
ALPHA=0.0; BETA=0.0; AR5=0.0; VELOCITY=0.2; DT_MIN=1; END_MIN=720
assert DT_MIN*60*VELOCITY<=20
print('Rainfall mm:',rain.sum(),'I30 mm/h:',i30,'Event EI30 MJ mm ha-1 h-1:',R_EVENT)
fig,ax=plt.subplots(figsize=(9,3)); ax.bar(minutes[1:],intensity,width=9,color='steelblue'); ax.set(xlabel='Minutes',ylabel='Intensity (mm/h)',title='Synthetic storm'); fig.savefig(WORK/'storm.png',dpi=180,bbox_inches='tight'); plt.show()
# NB00's K map uses kg h/(MJ mm); EI30 in MJ mm/(ha h)
# Confirm model R units from documentation before interpreting sediment magnitudes.
display(pd.DataFrame([dict(rainfall_mm=rain.sum(),EI30=R_EVENT,alpha=ALPHA,beta=BETA,antecedent_mm=AR5,velocity_m_s=VELOCITY,dt_min=DT_MIN,end_min=END_MIN)]))

## NB01.06 — Four factorial scenarios
CN values are deliberately synthetic, not a soil-group lookup or measured values. Cropland CN 80 vs 70, C-factor reduction 0 vs 40%. Fixed storm and other inputs. CN raster is 2D float32, zero outside model domain; river cells are overridden by model. Compare effects separately. The wrapper has an INI serialization issue for Endtime model, so patch that value explicitly after generating the file.
CN mode also requires Bulk density in Parameters Extensions; copy the existing reference value there without changing it.

In [ ]:
from pywatemsedem.scenario import Scenario
mask=np.asarray(catchment.mask.arr)==1
design=[('baseline',80,0.0),('CN_only',70,0.0),('C_only',80,0.4),('combined',70,0.4)]
scenarios={}; cn_maps={}
for nr,(label,cncrop,reduct) in enumerate(design,1):
    s=Scenario(catchment,2019,nr,choices)
    s.choices.extensions.curve_number=True
    s.choices.extensions.convert_output=False
    s.choices.parameters.r_factor=R_EVENT
    ep=s.choices.extensionparameters
    ep.alpha=ALPHA; ep.beta=BETA; ep.antecedent_rainfall=AR5
    ep.stream_velocity=VELOCITY; ep.desired_timestep=DT_MIN; ep.endtime_model=END_MIN
    s.choices.output.write_total_runoff=True
    s.choices.output.write_rainfall_excess=True
    p=gpd.read_file(inputdata_folder/'parcels.shp')
    p.loc[p.LANDUSE==-9999,'C_reduct']=reduct
    s.vct_parcels=p
    s.composite_landuse=s.create_composite_landuse()
    s.cfactor=s.create_cfactor(use_source_oriented_measures=reduct>0)
    s.ktc=s.create_ktc(ep.ktc_low.value,ep.ktc_high.value,ep.ktc_limit.value,not s.choices.extensions.create_ktc_map.value)
    lu=np.asarray(s.composite_landuse.arr)
    cn=np.zeros(lu.shape,dtype=np.float32)
    cn[mask & (lu>0)]=cncrop
    for cls,val in {-1:100,-2:98,-3:60,-4:69,-5:100,-6:69}.items():
        cn[mask & (lu==cls)]=val
    assert np.all((cn[mask]>0)&(cn[mask]<=100)),f'Unmapped land use: {np.unique(lu[mask & (cn==0)])}'
    s.cn=cn
    s.rainfall=rainfile
    s.prepare_input_files(); s.create_ini_file()
    cfg=configparser.ConfigParser(interpolation=None); cfg.optionxform=str
    cfg.read(s.ini)
    sec=next(x for x in cfg.sections() if x.lower()=='parameters extensions')
    key=next((x for x in cfg[sec] if x.lower()=='endtime model'),'Endtime model')
    cfg[sec][key]=str(END_MIN)
    # Pascal 5.0.3 reads BD from Parameters Extensions in CN mode,
    # whereas pywatemsedem always writes it under Parameters.
    bd=s.choices.parameters.bulk_density.value
    assert bd is not None and float(bd).is_integer() and float(bd)>0, f'Invalid bulk density: {bd}'
    bdkey=next((x for x in cfg[sec] if x.lower()=='bulk density'),'Bulk density')
    cfg[sec][bdkey]=str(int(bd))
    print(label,'CN bulk density (kg/m3):',int(bd))
    with s.ini.open('w') as f: cfg.write(f)
    scenarios[label]=s; cn_maps[label]=cn
    print(label,'CN range',cn[mask].min(),cn[mask].max(), 'INI',s.ini)
fig,axes=plt.subplots(2,4,figsize=(17,8),constrained_layout=True)
for i,(label,s) in enumerate(scenarios.items()):
    im=axes[0,i].imshow(np.ma.masked_where(~mask,cn_maps[label]),vmin=55,vmax=100,cmap='viridis'); axes[0,i].set_title(label)
    cm=axes[1,i].imshow(np.ma.masked_where(~mask,s.cfactor.arr),vmin=0,vmax=1,cmap='YlOrBr')
    for a in axes[:,i]: a.axis('off')
fig.colorbar(im,ax=list(axes[0]),label='CN'); fig.colorbar(cm,ax=list(axes[1]),label='C-factor')
fig.savefig(WORK/'scenario_inputs.png',dpi=180,bbox_inches='tight'); plt.show()
display(pd.DataFrame(design,columns=['scenario','cropland_CN','C_reduction_fraction']))

## NB01.07 — Run native CN-WS and check hydrological outputs
Logs and results stay local during calculation; each completed scenario is copied to Drive immediately for diagnostics.

In [ ]:
DEST=OUTPUT/RUN_ID; DEST.mkdir(exist_ok=True)
for label,s in scenarios.items():
    dest=s.sfolder.wsoutput_folder
    assert not any(dest.iterdir()), 'Use a new run to avoid stale output'
    p=subprocess.run([str(binary),str(s.ini)],capture_output=True,text=True,errors='replace')
    log=s.sfolder.scenario_folder/'model.log'; log.write_text(p.stdout+'\n'+p.stderr)
    print(label,p.returncode,p.stdout[-4000:],p.stderr[-3000:])
    shutil.copytree(s.sfolder.scenario_folder,DEST/label,dirs_exist_ok=True)
    p.check_returncode()
    required_outputs=['Discharge.txt','Total discharge.txt','Total runoff.rst','Remap.rst']
    missing=[x for x in required_outputs if not (dest/x).is_file()]
    assert not missing, f'{label}: missing CN outputs {missing}; inspect saved log'
    print(label,sorted(x.name for x in dest.iterdir()))

## NB01.08 — Hydrographs, outlet statistics and runoff maps
Report each outlet separately; do not sum nested outlets. Runoff raster is cumulative routed volume, not local generated water. Report native Remap values without assuming units because documentation and source descriptions differ.

In [ ]:
def read_report(path):
    df=pd.read_csv(path,sep='\t',skiprows=1,encoding='utf-8',encoding_errors='replace')
    return df.dropna(axis=1,how='all')
def numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(',','.',regex=False),errors='raise')
fig,axes=plt.subplots(2,2,figsize=(14,8),constrained_layout=True)
rows=[]
for ax,(label,s) in zip(axes.flat,scenarios.items()):
    d=s.sfolder.wsoutput_folder
    q=read_report(d/'Discharge.txt'); totals=read_report(d/'Total discharge.txt')
    t=numeric(q.iloc[:,0]); seconds=t*60 if 'min' in q.columns[0].lower() else t
    for j,col in enumerate(q.columns[1:],1):
        y=numeric(q[col]); ax.plot(seconds/60,y,label=col)
        # IDs correspond to the model's sequential Outlet headings.
        match=totals.loc[numeric(totals.iloc[:,0])==j]
        volume=float(numeric(match.iloc[:,1]).iloc[0]) if len(match)==1 else np.nan
        rows.append({'scenario':label,'outlet':col,'volume_m3':volume,'peak_m3_s':float(y.max()),'peak_time_min':float(seconds.iloc[y.argmax()]/60),'final_discharge_m3_s':float(y.iloc[-1])})
    ax.set(title=label,xlabel='Time (min)',ylabel='Discharge (m³/s)'); ax.legend(fontsize=7)
    for name in ['Sediment.txt','Sediment concentration.txt','Total sediment.txt']:
        if (d/name).exists(): print(label,name,(d/name).read_text(errors='replace')[:500] if name=='Total sediment.txt' else 'available')
fig.savefig(WORK/'hydrographs.png',dpi=180,bbox_inches='tight'); plt.show()
df=pd.DataFrame(rows); display(df); df.to_csv(WORK/'outlet_summary.csv',index=False)
if len(df):
    for outlet,g in df.groupby('outlet'):
        ax=g.set_index('scenario')[['volume_m3']].plot.bar(title=outlet+' | event runoff volume',rot=0,legend=False)
        ax.set_ylabel('m³'); plt.tight_layout(); plt.show()
for filename,title in [('Total runoff.rst','Cumulative routed runoff (m³/cell)'),('Remap.rst','Rainfall excess/deficit (native model values)')]:
    aa=[]
    for s in scenarios.values():
        with rasterio.open(s.sfolder.wsoutput_folder/filename) as ds:
            a=ds.read(1,masked=True); aa.append(np.ma.masked_where(~mask,a))
    lo=min(float(a.min()) for a in aa); hi=max(float(a.max()) for a in aa)
    fig,axes=plt.subplots(1,4,figsize=(16,5),constrained_layout=True)
    for ax,(label,a) in zip(axes,zip(scenarios,aa)):
        im=ax.imshow(a,vmin=lo,vmax=max(hi,lo+1e-9),cmap='viridis'); ax.set_title(label); ax.axis('off')
    fig.colorbar(im,ax=list(axes),label=title,shrink=.7); fig.savefig(WORK/(filename.replace('.rst','.png')),dpi=180,bbox_inches='tight'); plt.show()
shutil.copytree(WORK,DEST/'analysis',dirs_exist_ok=True,ignore=shutil.ignore_patterns('inputs','langegracht'))
print('Saved:',DEST)